# Cash transfers

This notebook reproduces Section 4.2 and Table 3 of the [AMIP paper](https://arxiv.org/pdf/2011.14999) using Python and JAX.

Requirements: R with the `foreign` package, and Python with JAX, NumPy, pandas, Patsy, SciPy, and IPython. The raw data must be available at `data/113289-V1-2/table1.dta` ([source archive](https://www.openicpsr.org/openicpsr/project/113289/version/V1/view)). No pre-existing processed CSV or RDS file is needed.

In [16]:
from pathlib import Path
from statistics import NormalDist
import shutil
import subprocess
import tempfile

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import pandas as pd
import patsy
from scipy.linalg import qr
from IPython.display import display

z = NormalDist().inv_cdf(0.975)

# A notebooks/data/ directory may also exist; identify the repository itself.
REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "notebooks/cash_transfers.ipynb").is_file()
     and (p / "data").is_dir()),
    None,
)
if REPO is None:
    raise FileNotFoundError("Run from the repository root or notebooks directory.")

DATA_FILE = REPO / "data/113289-V1-2/table1.dta"
EXPORT_DIR = REPO / "data/cash_transfers"
PROCESSED_FILE = EXPORT_DIR / "processed_CT_data.csv"


## Data Preparation

The export writes UTF-8 data to `data/cash_transfers/processed_CT_data.csv`, outside the downloaded source archive. Latin-1 Stata labels are explicitly converted to UTF-8. The export retains missing values and records a zero-based `source_row` before filtering, so selected observations can be traced back to the raw Stata file. The generated directory is ignored by Git and can be regenerated by rerunning these cells.

Adapted from Rachael Meager's data import and the cash-transfer sensitivity specification in the authors' [AMIP reproduction repository](https://github.com/rgiordan/AMIPPaper).

In [17]:
r_export = r"""
# Adapted from Rachael Meager's read-data-into-R.R (December 2018).
args <- commandArgs(trailingOnly = TRUE)
if (length(args) != 2L) {
  stop("Expected input .dta path and output CSV path.")
}
if (!requireNamespace("foreign", quietly = TRUE)) {
  stop("Install the R package 'foreign' before running this notebook.")
}

data <- foreign::read.dta(args[1], convert.factors = TRUE)
# The source archive uses Latin-1 labels (e.g. "sí"). Declare their encoding
# before writing UTF-8; otherwise write.csv can truncate invalid labels.
data[] <- lapply(data, function(x) {
  if (is.factor(x)) {
    levels(x) <- iconv(levels(x), from = "latin1", to = "UTF-8")
  } else if (is.character(x)) {
    x <- iconv(x, from = "latin1", to = "UTF-8")
  }
  x
})
required <- c(
  "Cind", "t", "treatp", "treatnp", "avg_shock", "hhhage", "hhhsex",
  "p16", "hhhalpha", "hhhspouse", "yycali_1", "region_1", "indice",
  "hectareas", "vhhnum", "village"
)
missing_columns <- setdiff(required, names(data))
if (length(missing_columns)) {
  stop(paste("Missing source columns:", paste(missing_columns, collapse = ", ")))
}

# Preserve original ordering and factor labels; filtering follows in Python.
data$source_row <- seq_len(nrow(data)) - 1L
dir.create(dirname(args[2]), recursive = TRUE, showWarnings = FALSE)
write.csv(data, args[2], row.names = FALSE, na = "NA", fileEncoding = "UTF-8")
cat(sprintf("Exported %d source observations.\n", nrow(data)))
"""


In [18]:
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Extract the source archive so this file exists: {DATA_FILE}")
rscript = shutil.which("Rscript")
if rscript is None:
    raise FileNotFoundError("Rscript is not on PATH. Install R to run the embedded export.")

with tempfile.TemporaryDirectory() as tmp:
    script = Path(tmp) / "export.R"
    script.write_text(r_export, encoding="utf-8")
    # Check the export before replacing a previously generated dataset.
    exported_csv = Path(tmp) / "processed_CT_data.csv"
    result = subprocess.run(
        [rscript, str(script), str(DATA_FILE), str(exported_csv)],
        check=True, capture_output=True, text=True,
    )
    raw_df = pd.read_csv(exported_csv, encoding="utf-8")
    assert raw_df["source_row"].is_unique
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(exported_csv, PROCESSED_FILE)
    print(result.stdout.strip())

print(f"Generated data: {PROCESSED_FILE.relative_to(REPO)}")


Exported 65876 source observations.
Generated data: data/cash_transfers/processed_CT_data.csv


Import the exported data and apply the paper's `Cind < 10000` trimming. Treatment columns are missing outside their corresponding poor/non-poor groups. Each regression also removes observations missing a required outcome or control, as determined by Patsy. Village identifiers are required for clustered standard errors.

In [19]:
df = pd.read_csv(PROCESSED_FILE, encoding="utf-8")
df = df.loc[df["Cind"] < 10000].copy()

for treatment in ("treatp", "treatnp"):
    assert set(df[treatment].dropna().unique()) == {0, 1}

print(f"Raw observations: {len(raw_df):,}")
print(f"Observations after consumption trimming: {len(df):,}")


Raw observations: 65,876
Observations after consumption trimming: 59,455


## Model Definition

For each poor/non-poor group and period $t\in\{8,9,10\}$, estimate

$$C\_ind_{it} = \beta_0 + \beta_1 treat_{poor,i} + \beta_2 treat_{nonpoor,i} + \sum_{k=1}^{K} \beta_{2+k} X_{itk} + \epsilon_{it}.$$

To reproduce Table 3, we estimate separate regressions for poor and non-poor households within each period, following the authors' replication code. Each regression includes the corresponding treatment indicator and allows its own intercept and control coefficients.

$$Cind_i=\beta_0+\beta_1 treatment_i+X_i^\top\gamma+\epsilon_i$$

In [20]:
base_formula = (
    "Cind ~ {treatment} + avg_shock + hhhage + C(hhhsex) + C(p16) "
    "+ C(hhhalpha) + C(hhhspouse) + yycali_1 + C(region_1) "
    "+ indice + hectareas + vhhnum"
)


def full_rank_columns(X):
    """Keep an independent basis containing treatment (column 0)."""
    _, R, piv = qr(np.asarray(X), mode="economic", pivoting=True)
    rank = np.count_nonzero(np.abs(np.diag(R)) > 1e-10 * abs(R[0, 0]))
    keep = np.sort(piv[:rank])
    if 0 not in keep:
        raise ValueError("Treatment coefficient is not identified.")
    return keep


def prepare_cash_data(treatment, period):
    subset = df.loc[(df["t"] == period) & df[treatment].notna()].copy()
    yp, xp = patsy.dmatrices(
        base_formula.format(treatment=treatment), subset, return_type="dataframe"
    )
    d = subset.loc[xp.index].copy()
    assert d["village"].notna().all()

    columns = [treatment] + [c for c in xp if c != treatment]
    X = xp[columns].to_numpy(float)
    scale = np.sqrt(np.mean(X**2, axis=0))
    scale[scale == 0] = 1
    scale[0] = 1
    X = X / scale
    X = X[:, full_rank_columns(X)]
    return d, X, yp.to_numpy().ravel()


In [21]:
expected_samples = {
    ("treatp", 8): 10936, ("treatp", 9): 9630, ("treatp", 10): 10518,
    ("treatnp", 8): 4624, ("treatnp", 9): 3838, ("treatnp", 10): 4266,
}
sample_sizes = []
for (treatment, period), expected in expected_samples.items():
    case_data, _, _ = prepare_cash_data(treatment, period)
    assert len(case_data) == expected, (treatment, period, len(case_data), expected)
    sample_sizes.append({"treatment": treatment, "period": period, "N": len(case_data)})
display(pd.DataFrame(sample_sizes))


,treatment,period,N
0,treatp,8,10936
1,treatp,9,9630
2,treatp,10,10518
3,treatnp,8,4624
4,treatnp,9,3838
5,treatnp,10,4266


In [22]:
def fit_ols(w, X, y, clusters, num_clusters):
    """Return the treatment coefficient and village-clustered HC0 standard error."""
    A = X.T @ (w[:, None] * X)
    beta = jnp.linalg.solve(A, X.T @ (w * y))
    residual = y - X @ beta

    # Weighted observation scores, summed within villages.
    scores = X * (w * residual)[:, None]
    grouped = jax.ops.segment_sum(scores, clusters, num_segments=num_clusters)
    # Match zaminfluence's centered cluster scores.
    grouped = grouped - jnp.mean(grouped, axis=0, keepdims=True)
    bread = jnp.linalg.inv(A)
    covariance = bread @ (grouped.T @ grouped) @ bread.T

    return jnp.array([beta[0], jnp.sqrt(covariance[0, 0])])


## Model Fitting: Poor Households, Period 8

Begin with one case before running the full analysis, as in the Mexico-profit example in `microcredit.ipynb`. The expected baseline coefficient and standard error are **17.312 (4.576)**. Unlike microcredit, the standard error comes from the village-clustered sandwich rather than a homoskedastic residual variance.

In [23]:
d, X, y = prepare_cash_data("treatp", 8)
X, y = jnp.asarray(X), jnp.asarray(y)
cluster_codes, villages = pd.factorize(d["village"], sort=True)
clusters = jnp.asarray(cluster_codes)
num_clusters = len(villages)
w = jnp.ones(len(d))

# A function of deletion weights alone, retaining village clustering.
fit = jax.jit(lambda weights: fit_ols(weights, X, y, clusters, num_clusters))
b, se = np.asarray(fit(w))
np.testing.assert_allclose(np.round([b, se], 3), [17.312, 4.576], atol=1e-12)
print(f"Estimate: {b:.3f}")
print(f"Standard error: {se:.3f}")


Estimate: 17.312
Standard error: 4.576


## Differentiation

Differentiate both the coefficient and its clustered standard error with respect to every observation's deletion weight. This includes changes in the fitted residuals, the design inverse, and village score sums. Individual observations are removed; villages define the uncertainty calculation, not the deletion unit.

The two derivative rows are $db_i=\partial\hat\beta_1/\partial w_i$ and $ds_i=\partial SE_1/\partial w_i$, evaluated at all-one weights. A central finite-difference check verifies the derivatives on observations with large influence.

In [24]:
derivatives = np.asarray(jax.jacrev(fit)(w))

db, ds = derivatives

print(f"Derivative shape: {derivatives.shape} (coefficient/SE by observation)")


Derivative shape: (2, 10936) (coefficient/SE by observation)


## Conclusion Targets

Let $L=b-zSE$ and $U=b+zSE$, where $z=\Phi^{-1}(0.975)$. Each target has its own influence vector:

- **Sign change:** move $b$ through zero.
- **Significance change:** move the confidence-interval endpoint closest to zero through zero. This also allows an initially insignificant result to become significant.
- **Significant sign change:** move $U$ below zero for positive $b$, or $L$ above zero for negative $b$.

The endpoint derivatives are $db-z\,ds$ and $db+z\,ds$. Holding the standard error fixed would not reproduce Table 3.

In [25]:
def make_targets(b, se, db, ds):
    L, U = b - z * se, b + z * se
    return {
        "Sign change": (b, db),
        "Significance change": (
            (L, db - z * ds) if abs(L) < abs(U) else (U, db + z * ds)
        ),
        "Significant sign change": (
            (U, db + z * ds) if b >= 0 else (L, db - z * ds)
        ),
    }

## Rank Observations and Predicted Deletion Count

For scalar target $q$ and influence scores $\psi_i$, deleting a subset $S$ predicts

$$q_{after}\approx q-\sum_{i\in S}\psi_i.$$

Rank positive gains $\operatorname{sign}(q)\psi_i$ and take the shortest prefix whose cumulative gain reaches $|q|$. The resulting count is the minimum predicted by the linear approximation, not a proof of the globally minimal actual deletion subset.

In [26]:
def select_subset(q, psi):
    gain = np.sign(q) * np.asarray(psi)
    order = np.argsort(-gain, kind="stable")
    order = order[gain[order] > 0]
    hits = np.flatnonzero(np.cumsum(gain[order]) >= abs(q))
    return None if len(hits) == 0 else order[:int(hits[0]) + 1]


example_counts = []
for target, (q, psi) in make_targets(b, se, db, ds).items():
    drop = select_subset(q, psi)
    example_counts.append({"target": target, "predicted_deletions": len(drop)})
display(pd.DataFrame(example_counts))


,target,predicted_deletions
0,Sign change,252
1,Significance change,83
2,Significant sign change,464


## Refit and Check the Actual Change

Set the selected weights to zero and refit. If a deletion removes a factor level, rebuild the full-rank basis and retained village codes, matching the authors' row-deletion refit. Record the actual coefficient, clustered standard error, and whether the target was achieved; do not increase the deletion count just to make an approximate prediction succeed.

In [27]:
def analyse_case(treatment, period):
    d, X, y = prepare_cash_data(treatment, period)

    X, y = jnp.asarray(X), jnp.asarray(y)
    
    cluster_codes, villages = pd.factorize(d["village"], sort=True)
    clusters = jnp.asarray(cluster_codes)
    num_clusters = len(villages)
    w = jnp.ones(len(d))
    fit = jax.jit(lambda weights: fit_ols(weights, X, y, clusters, num_clusters))

    b, se = np.asarray(fit(w))
    db, ds = np.asarray(jax.jacrev(fit)(w))

    records = []

    for target, (q, psi) in make_targets(b, se, db, ds).items():
        drop = select_subset(q, psi)

        if drop is None:
            raise RuntimeError(f"Linear approximation cannot reach {target}.")

        modified = w.at[drop].set(0.0)
        retained = np.ones(len(y), dtype=bool)
        retained[drop] = False
        cols = full_rank_columns(np.asarray(X)[retained])
        if len(cols) == X.shape[1]:
            br, sr = np.asarray(fit(modified))
        else:
            rc, rn = pd.factorize(d["village"].to_numpy()[retained], sort=True)
            br, sr = np.asarray(fit_ols(
                jnp.ones(retained.sum()), X[retained][:, cols], y[retained],
                jnp.asarray(rc), len(rn),
            ))

        significant_after = abs(br) > z * sr
        achieved = {
            "Sign change": b * br < 0,
            "Significance change": significant_after != (abs(b) > z * se),
            "Significant sign change": (b * br < 0) and significant_after,
        }[target]
        records.append({
            "group": "Poor" if treatment == "treatp" else "Non-poor",
            "period": period,
            "N": len(y),
            "estimate": b,
            "se": se,
            "target": target,
            "dropped": len(drop),
            "percent": 100 * len(drop) / len(y),
            "predicted_target": q - np.asarray(psi)[drop].sum(),
            "refit_estimate": br,
            "refit_se": sr,
            "achieved": bool(achieved),
            "dropped_row_ids": d["source_row"].iloc[drop].tolist(),
        })
    return records


example_results = pd.DataFrame(analyse_case("treatp", 8))
display(example_results.drop(columns="dropped_row_ids").round(3))


,group,period,N,estimate,se,target,dropped,percent,predicted_target,refit_estimate,refit_se,achieved
0,Poor,8,10936,17.312,4.576,Sign change,252,2.304,-0.033,-0.656,3.745,True
1,Poor,8,10936,17.312,4.576,Significance change,83,0.759,-0.001,7.284,4.087,True
2,Poor,8,10936,17.312,4.576,Significant sign change,464,4.243,-0.001,-7.212,3.443,True


## All Groups and Periods

Apply the same pipeline to poor and non-poor households in periods 8, 9, and 10. Each case uses its own complete-case sample and village encoding.

In [28]:
cash_results = pd.DataFrame([
    row
    for treatment in ("treatp", "treatnp")
    for period in (8, 9, 10)
    for row in analyse_case(treatment, period)
])
display(cash_results.drop(columns="dropped_row_ids").round(3))


,group,period,N,estimate,se,target,dropped,percent,predicted_target,refit_estimate,refit_se,achieved
0,Poor,8,10936,17.312,4.576,Sign change,252,2.304,-0.033,-0.656,3.745,True
1,Poor,8,10936,17.312,4.576,Significance change,83,0.759,-0.001,7.284,4.087,True
2,Poor,8,10936,17.312,4.576,Significant sign change,464,4.243,-0.001,-7.212,3.443,True
3,Poor,9,9630,27.924,5.770,Sign change,345,3.583,-0.010,-1.377,4.406,True
4,Poor,9,9630,27.924,5.770,Significance change,146,1.516,-0.041,7.077,4.555,True
5,Poor,9,9630,27.924,5.770,Significant sign change,588,6.106,-0.021,-8.951,4.251,True
6,Poor,10,10518,33.861,4.468,Sign change,697,6.627,-0.011,-2.559,3.541,True
7,Poor,10,10518,33.861,4.468,Significance change,435,4.136,-0.001,4.806,3.684,True
8,Poor,10,10518,33.861,4.468,Significant sign change,986,9.374,-0.018,-9.416,3.296,True
9,Non-poor,8,4624,-5.444,7.133,Sign change,5,0.108,0.205,0.260,6.410,True
